# Exportar adapter treinado — sem gastar créditos de treino

Este notebook **não treina nada**. Ele só ajuda a tirar o adapter (já treinado, salvo em
`Meu Drive/outputs-logos-1-v2/adapter`) do Colab pra você testar localmente via o harness
(`scripts/run_eval.py --adapter ...` ou um script de chat interativo).

**Antes de rodar qualquer célula, leia isto — é o jeito mais barato de fazer o que você quer:**

> Os arquivos do adapter (`adapter_config.json`, `adapter_model.safetensors`,
> `tokenizer.json`, etc.) já estão prontos no seu Google Drive, visíveis pela interface web.
> Um adapter LoRA é pequeno (tipicamente dezenas a poucas centenas de MB, não o tamanho do
> modelo de 8B inteiro) — **você pode simplesmente selecionar os 6 arquivos na página do Drive
> e clicar em "Baixar", sem gastar um único crédito de Colab.** A Seção 1 abaixo automatiza
> isso (zipa tudo num arquivo só), mas ela é só conveniência — não é obrigatória.

As seções 2 e 3 abaixo (sanity-check e merge) **precisam de GPU** e consomem créditos de
verdade — são opcionais, marcadas com custo estimado, rode só se quiser confirmar/mesclar
antes de baixar.

## 0. Ambiente — detecta Colab/Kaggle/local (mesma lógica do notebook de treino)

In [ ]:
import os


def _running_on_kaggle() -> bool:
    return bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))


def _running_on_colab() -> bool:
    if _running_on_kaggle():
        return False
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


ON_COLAB = _running_on_colab()
ON_KAGGLE = _running_on_kaggle()
print(f"Colab: {ON_COLAB} | Kaggle: {ON_KAGGLE} | local: {not (ON_COLAB or ON_KAGGLE)}")

## 1. Baixar só o adapter (custo: zero GPU, poucos segundos)

**Runtime recomendado: SEM GPU** (Ambiente de execução → Alterar tipo de ambiente de
execução → CPU). Este passo só mexe em arquivo, não carrega nenhum modelo — rodar numa GPU
alocada aqui seria desperdiçar crédito à toa.

Ajuste `ADAPTER_DIR` se o caminho no seu Drive for diferente do que aparece no screenshot
(`Meu Drive/outputs-logos-1-v2/adapter`).

In [ ]:
ADAPTER_DIR = "/content/drive/MyDrive/outputs-logos-1-v2/adapter"  # ajuste se necessário

if ON_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
else:
    print("Não está no Colab — pule esta célula e acesse o Drive direto pelo navegador.")

In [ ]:
import shutil
from pathlib import Path

adapter_path = Path(ADAPTER_DIR)
assert adapter_path.is_dir(), f"pasta não encontrada: {adapter_path} — confira ADAPTER_DIR acima"

files = sorted(p.name for p in adapter_path.iterdir())
print(f"{len(files)} arquivos em {adapter_path}:")
for f in files:
    size_mb = (adapter_path / f).stat().st_size / (1024 * 1024)
    print(f"  {f:32s} {size_mb:8.2f} MB")

zip_base = "/content/logos3_adapter"
zip_path = shutil.make_archive(zip_base, "zip", root_dir=str(adapter_path))
print(f"\nZip pronto: {zip_path} ({Path(zip_path).stat().st_size / (1024*1024):.2f} MB)")

In [ ]:
# Baixa o zip pro seu computador. Se o Colab reclamar de arquivo grande / a conexão cair no
# meio, é mais confiável copiar pro Drive (célula seguinte) e baixar pela interface web do
# Drive em vez do download direto do notebook.
if ON_COLAB:
    from google.colab import files

    files.download(zip_path)

In [ ]:
# Alternativa mais robusta pra arquivos grandes: copia o zip pro Drive (rápido, é cópia
# dentro do próprio Google) e você baixa depois pela interface web do Drive, sem depender da
# conexão do notebook ficar de pé até o fim do download.
import shutil as _shutil

drive_copy = "/content/drive/MyDrive/logos3_adapter.zip"
if ON_COLAB:
    _shutil.copy(zip_path, drive_copy)
    print(f"Copiado pra {drive_copy} — baixe pela interface web do Drive quando quiser.")

## 2. (Opcional) Sanity-check — confirma que o adapter carrega e gera algo plausível

**Runtime: GPU (L4/T4). Custo: baixo** (carrega o modelo base quantizado em 4-bit + o
adapter, gera poucos tokens). Pule esta seção se só quer baixar os arquivos — ela existe pra
você confirmar ANTES de testar localmente que o adapter não está corrompido/incompleto.

In [ ]:
%pip install -q transformers==5.10.2 peft==0.19.1 bitsandbytes==0.49.2 accelerate==1.10.1 "torchao>=0.16.0" 

In [ ]:
import os

# expandable_segments evita fragmentação de VRAM — precisa ser setado ANTES de importar torch.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL = "ibm-granite/granite-4.1-8b"  # bate com configs/train_l4.yaml

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=quant_config, device_map="auto"
)
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.eval()
print("Adapter carregado sobre o modelo base — ok.")

In [ ]:
# Mesmo formato de prompt usado no treino/avaliação (PRAXIS_SYSTEM_PROMPT + [USER]/[ASSISTANT])
# — testar com um prompt de formato diferente do treinado não prova nada sobre o adapter.
SYSTEM_PROMPT = (
    "Você é Logos-3, um agente de engenharia de software da família de modelos Conatus. "
    "Raciocine em <think>, use "
    '<tool_call name="..."> quando precisar de uma ferramenta, e responda ao usuário só '
    "dentro de <final>."
)
prompt = f"{SYSTEM_PROMPT}\n\n[USER]\nquanto é 2+2?\n\n[ASSISTANT]\n"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    output = model.generate(
        **inputs, max_new_tokens=80, do_sample=False, repetition_penalty=1.15,
        pad_token_id=tokenizer.pad_token_id,
    )
generated = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print(repr(generated))

In [ ]:
# Libera VRAM antes de qualquer célula seguinte (merge, seção 3) — sem isso o modelo desta
# seção continua ocupando a GPU.
del model, base_model
import gc

gc.collect()
torch.cuda.empty_cache()
print(f"VRAM alocada: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## 3. (Opcional, mais caro) Mesclar adapter + base num modelo único standalone

**Runtime: GPU (L4, precisa de ~16-20GB livres). Custo: alto** (carrega o modelo base em
bfloat16 — não quantizado, precisa de precisão cheia pra mesclar os pesos LoRA corretamente
— e salva um modelo standalone de ~16GB). Só rode isso se quiser testar localmente SEM
precisar do `peft` instalado nem apontar `--adapter` toda vez — um único modelo pronto pra
`from_pretrained` direto. Pra maioria dos casos, a Seção 1 (só o adapter) já é suficiente e
muito mais barata — o harness (`TransformersModelRunner`) já sabe carregar base+adapter
juntos localmente.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

base_model_fp = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=torch.bfloat16, device_map="auto"
)
merge_model = PeftModel.from_pretrained(base_model_fp, ADAPTER_DIR)
merged = merge_model.merge_and_unload()  # aplica o delta LoRA nos pesos base, sem depender de peft depois

MERGED_DIR = "/content/logos3_merged"
merged.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)
print(f"Modelo mesclado salvo em {MERGED_DIR}")

In [ ]:
# Modelo grande (~16GB) — copiar pro Drive é mais confiável que files.download direto.
# Depois baixe pela interface web do Drive (retomável, não depende da sessão do notebook
# ficar viva até o fim de um download de vários GB).
import shutil

if ON_COLAB:
    merged_zip_base = "/content/drive/MyDrive/logos3_merged"
    shutil.make_archive(merged_zip_base, "zip", root_dir=MERGED_DIR)
    print(f"Zipado direto no Drive: {merged_zip_base}.zip — baixe pela interface web quando quiser.")

## 3.5. (Recomendado) Converter pra GGUF e rodar via Ollama — mais leve que Transformers local

**Runtime: pode ser SEM GPU (CPU já basta — a conversão/quantização não usa GPU nenhuma).
Custo: baixo**, poucos minutos de CPU. Diferente da Seção 3 (que salva um modelo `bfloat16`
de ~16GB), esta seção produz um `.gguf` quantizado de ~5GB, pronto pro Ollama — muito mais
leve pra rodar localmente (o Ollama não precisa de VRAM de GPU dedicada nem de
`transformers`/`peft`/`bitsandbytes` instalados no seu PC).

**Se você já rodou a Seção 3 nesta mesma sessão do Colab**, pode pular direto pra célula de
conversão abaixo (`MERGED_DIR` já existe em memória). **Se está numa sessão NOVA** (voltou ao
Colab depois de já ter baixado o modelo mesclado), a célula seguinte retoma a partir do zip
que a Seção 3 já deixou no seu Drive — sem precisar re-mesclar do zero.

In [ ]:
# Retoma a partir do Drive se MERGED_DIR não existir em memória (sessão nova do Colab).
import os
from pathlib import Path

MERGED_DIR = globals().get("MERGED_DIR", "/content/logos3_merged")

if not Path(MERGED_DIR).is_dir():
    if ON_COLAB:
        from google.colab import drive

        drive.mount("/content/drive", force_remount=False)
    drive_zip = "/content/drive/MyDrive/logos3_merged.zip"
    assert Path(drive_zip).is_file(), (
        f"Não achei {MERGED_DIR} em memória nem {drive_zip} no Drive — rode a Seção 3 primeiro."
    )
    import shutil

    shutil.unpack_archive(drive_zip, MERGED_DIR)
    print(f"Modelo mesclado restaurado do Drive em {MERGED_DIR}")
else:
    print(f"Usando {MERGED_DIR} já em memória desta sessão.")

In [ ]:
# Clona o llama.cpp (repo leve, sem pesos de modelo) e instala as dependências de conversão.
!git clone --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
%pip install -q -r /content/llama.cpp/requirements.txt

In [ ]:
# Converte HF -> GGUF em f16 (ainda não quantizado, ~16GB — passo intermediário, não é o
# arquivo final que você vai baixar). É I/O-bound (lê e reescreve os pesos), não usa GPU.
GGUF_F16 = "/content/logos3.f16.gguf"
!python /content/llama.cpp/convert_hf_to_gguf.py "{MERGED_DIR}" --outfile "{GGUF_F16}" --outtype f16

import os
print(f"\nGGUF f16: {os.path.getsize(GGUF_F16) / 1e9:.2f} GB")

**Se a célula acima falhar com algo tipo "Model architecture not supported"**: a arquitetura
do Granite 4.1 pode não estar coberta pela versão do llama.cpp clonada (arquiteturas novas
demoram a ganhar suporte). Como você já tem `granite4.1:8b` funcionando no seu Ollama local
(`ollama list` mostrou isso), o suporte EXISTE em algum lugar — tente atualizar o clone
(remover `--depth 1` acima, ou apontar pro branch/tag mais recente) antes de desistir dessa
rota. Se continuar falhando, me mostra o erro exato que eu ajudo a diagnosticar.

In [ ]:
# Builda só o binário llama-quantize (não o projeto inteiro, mais rápido). Colab já vem com
# cmake/gcc no Ubuntu base, sem precisar instalar nada extra.
!cmake -S /content/llama.cpp -B /content/llama.cpp/build -DCMAKE_BUILD_TYPE=Release
!cmake --build /content/llama.cpp/build --config Release -j --target llama-quantize

In [ ]:
# Quantiza pra Q4_K_M (bom equilíbrio tamanho/qualidade — mesmo nível do logos-v2 que você já
# tem rodando). Troque por Q5_K_M ou Q8_0 se quiser mais qualidade à custa de mais espaço.
GGUF_Q4 = "/content/logos3.Q4_K_M.gguf"
!/content/llama.cpp/build/bin/llama-quantize "{GGUF_F16}" "{GGUF_Q4}" Q4_K_M

import os
print(f"\nGGUF quantizado: {os.path.getsize(GGUF_Q4) / 1e9:.2f} GB")

In [ ]:
# Copia pro Drive (mais confiável pra ~5GB que download direto do notebook) e também tenta o
# download direto — use o que funcionar primeiro.
import shutil

if ON_COLAB:
    drive_gguf = "/content/drive/MyDrive/logos3.Q4_K_M.gguf"
    shutil.copy(GGUF_Q4, drive_gguf)
    print(f"Copiado pro Drive: {drive_gguf} — baixe pela interface web quando quiser.")

    from google.colab import files

    files.download(GGUF_Q4)

### Depois de baixar o `.gguf`, no seu PC (sem precisar de GPU nem de Python/transformers):

```powershell
# Modelfile mínimo — o harness sempre manda o prompt em modo "raw" (ver src/inference/
# ollama_runner.py), então o Ollama NUNCA usa TEMPLATE/SYSTEM daqui; não precisa deles.
"FROM ./logos3.Q4_K_M.gguf" | Out-File -Encoding utf8 Modelfile

ollama create logos3 -f Modelfile
ollama list   # confirma que "logos3" apareceu
```

Depois teste com o chat interativo do projeto (agora com suporte a Ollama):

```powershell
python scripts/chat_cli.py --ollama-model logos3
```

**Importante**: rodar `ollama run logos3` diretamente (sem passar por `chat_cli.py`) testa só
o texto que o modelo gera cru — nenhum `<tool_call>` é executado de verdade, o modelo pode até
"inventar" um `<tool_result>` sem ninguém ter rodado nada. Pra testar com ferramentas
EXECUTANDO de verdade (o que importa pra avaliar o adapter), use sempre o `chat_cli.py`.

## 4. Testar localmente (depois de baixar)

Depois de baixar (Seção 1, adapter só — recomendado) pra uma pasta local, ex.
`C:\Users\Pichau\logos3-adapter\`:

```bash
pip install -r requirements-train.txt
python scripts/run_eval.py --adapter "C:\Users\Pichau\logos3-adapter" --base-model ibm-granite/granite-4.1-8b --device cuda
```

Sem GPU local, `--device cpu` funciona mas é bem mais lento (útil como smoke-test de que o
adapter carrega, não pra avaliar qualidade real — mesma ressalva de D-eval-fase-g-probes no
`docs/PLAN.md`).

Se mesclou na Seção 3, é o mesmo comando trocando `--base-model` pelo caminho local do
modelo mesclado e **sem** passar `--adapter` — mas hoje `run_eval.py` exige `--adapter` como
argumento obrigatório (grupo mutuamente exclusivo com `--demo`), então testar um modelo já
mesclado localmente precisa de um ajuste pequeno nesse script (ou de um script de chat
interativo separado). Se quiser, posso escrever esse script de chat livre pra CLI na próxima
mensagem — meu foco aqui foi só te dar como baixar o adapter/modelo gastando o mínimo de
crédito possível.